In [1]:
%%html
<script>
  function code_toggle() {
    if (code_shown){
      $('div.input').hide('500');
      $('#toggleButton').val('Show Code')
    } else {
      $('div.input').show('500');
      $('#toggleButton').val('Hide Code')
    }
    code_shown = !code_shown
  }

  $( document ).ready(function(){
    code_shown=false;
    $('div.input').hide()
  });
</script>
<form action="javascript:code_toggle()"><input type="submit" id="toggleButton" value="Show Code"></form>

<!---
Latex Macros
-->
$$
\newcommand{\Xs}{\mathcal{X}}
\newcommand{\Ys}{\mathcal{Y}}
\newcommand{\y}{\mathbf{y}}
\newcommand{\a}{\mathbf{a}}
\newcommand{\repr}{\mathbf{f}}
\newcommand{\repry}{\mathbf{g}}
\newcommand{\x}{\mathbf{x}}
\newcommand{\vocab}{V}
\newcommand{\params}{\boldsymbol{\theta}}
\newcommand{\param}{\theta}
\DeclareMathOperator{\perplexity}{PP}
\DeclareMathOperator{\argmax}{argmax}
\DeclareMathOperator{\argmin}{argmin}
\newcommand{\train}{\mathcal{D}}
\newcommand{\counts}[2]{\#_{#1}(#2) }
\newcommand{\indi}{\mathbb{I}}
$$

In [1]:
%%capture
%load_ext autoreload
%autoreload 2
%matplotlib inline
# preamble
import sys
sys.path.append("..")
import statnlpbook.transition as transition

# Transition-based dependency parsing

This chapter is influenced by the EACL 2014 tutorial by Ryan McDonald and Joakim Nivre.

In the [constituency-parsing material](../exercises/parsing.ipynb) we saw how to develop a syntactic parser based on context-free grammars (CFG). In this chapter we will see how to develop a syntactic parser based on a different paradigm, dependency parsing.

The key idea in dependency parsing is that syntactic structure of lexical items, linked by binary
asymmetric relations called dependencies [[Nivre, 2008]](http://www.mitpressjournals.org/doi/pdf/10.1162/coli.07-056-R1-07-027). More simply, syntax is represented as directed edges between words, commonly referred to arcs in the literature. Thus unlike the trees in [CFG parsing](../exercises/parsing.ipynb), dependency trees have only terminal nodes (the words of the sentences), which can appear as leaves as and non-leaf nodes. Here is the dependency graph for the sentence:

> "Economic news had little effect on financial markets"

In [ ]:
tokens = ["Economic", "news", "had", "little", "effect", "on", "financial", "markets", "."]
arcs = {(1,0,"amod"), (2,1,"nsubj"), (2, 4, "dobj"), (4,3,"amod"), (4,5, "prep"), (5,7,"pmod"), (7,6,"amod")}

#transition.render_tree(tokens, arcs)

transition.render_displacy(*transition.to_displacy_graph(arcs, tokens))

<Dependency diagram>

Following [Nivre (2008)](https://aclanthology.org/J08-4003/), a dependency parse is a graph $\y=(\x,\a)$ where:

* $\x=\{0,1,\ldots,N\}$ contains one node per word and the artificial `ROOT` node 0.
* $\a\subseteq\x\times\x\times L$ contains labeled, directed head-to-dependent arcs.

A valid dependency tree must be:

* **rooted:** `ROOT` has no incoming arc;
* **single-headed:** every other node has exactly one incoming arc;
* **acyclic:** no directed cycle exists;
* **connected:** there is an undirected path between every pair of nodes.

The example above leaves the period unattached, so it is a dependency **forest**, not a complete tree. A complete analysis attaches punctuation to an appropriate head and connects the sentence's syntactic root to the artificial `ROOT` node.

In [3]:
tokens = ["ROOT", "Alice", "saw", "Bob"]
arcs = {(0,2, "root"), (2,1,"nsubj"), (2,3,"dobj")}

transition.render_displacy(*transition.to_displacy_graph(arcs, tokens))

A straightforward approach to dependency parsing would be given a sentence \\(\x\\), to enumerate over all valid graphs for the sentence $\y\in\Ys_x$ and score them using an appropriate function \\(s_\params(\x,\y)\\), a structured-prediction problem. While such an approach is possible and there has been a lot of work often referred to as graph-based parsing, e.g. [McDonald et al. (2005)](https://aclanthology.org/H05-1066/), in this note we will focus on transition-based approaches, which decompose the task into a sequence of label predictions that can be learned with a classifier.

We use the **arc-standard** transition system.

**Configuration:** a stack $S$, a buffer $B$ of unprocessed words, and the set $A$ of arcs built so far. The stack top is written on the right; the buffer front is written on the left.

* **Initial:** $S=[\mathrm{ROOT}]$, $B$ contains all words in order, and $A=\varnothing$.
* **Terminal:** $B$ is empty and $S=[\mathrm{ROOT}]$.

**Actions:**

* **SHIFT:** move the buffer front onto the stack:
  $(S,i|B,A)\rightarrow(S|i,B,A)$.
* **LEFT-ARC-$r$:** attach the second stack item $i$ to the top item $j$, then remove $i$:
  $(S|i|j,B,A)\rightarrow(S|j,B,A\cup\{(j,i,r)\})$.
* **RIGHT-ARC-$r$:** attach the top stack item $j$ to the second item $i$, then remove $j$:
  $(S|i|j,B,A)\rightarrow(S|i,B,A\cup\{(i,j,r)\})$.

SHIFT requires a non-empty buffer. Arc actions require two stack items and must not make `ROOT` a dependent. Each word is shifted once and removed by an arc action once: **$2N$ actions** for $N$ words.

The small implementation below assumes a valid action sequence. A learned parser must mask illegal actions and stop only at a terminal configuration.

In [4]:
from collections import deque

class Configuration():
    def __init__(self, tokenized_sentence):
        # This implements the initial configuration for a sentence
        self.arcs = set()
        self.buffer = deque()
        self.sentence = tokenized_sentence
        for idx, token in enumerate(tokenized_sentence[1:], start=1):
            self.buffer.append({"index": idx, "form": token})
        self.stack = [{"index": 0, "form": "ROOT"}]
        
import copy
def parse(tokenized_sentence, actions):
    # This stores the (configuration, action) tuples generated
    transitions = []
    
    # Initialize the configuration
    configuration = Configuration(tokenized_sentence)
    transitions.append((copy.deepcopy(configuration), ""))
    
    for action in actions:
        if action == "shift":
            token = configuration.buffer.popleft()
            configuration.stack.append(token)
        elif action.startswith("leftArc"):
            head = configuration.stack[-1]
            dependent = configuration.stack.pop(-2)
            label = action.split("-")[1]
            configuration.arcs.add((int(head["index"]), int(dependent["index"]), label))
        elif action.startswith("rightArc"):
            head = configuration.stack[-2]
            dependent = configuration.stack.pop()
            label = action.split("-")[1]
            configuration.arcs.add((int(head["index"]), int(dependent["index"]), label))            
        
        transitions.append((copy.deepcopy(configuration), action))
    
    if len(configuration.buffer) == 0 and len(configuration.stack) <= 1:
        transitions.append((copy.deepcopy(configuration), ""))
    return transitions

Let's see how we can parse the example sentence using this transition system defined above assuming we are given the correct sequence of actions:

In [5]:
tokenized_sentence = ["ROOT", "Alice", "saw", "Bob"]
actions = ["shift","shift", "leftArc-nsubj", "shift", "rightArc-dobj", "rightArc-root"]

transitions = parse(tokenized_sentence, actions)

transition.render_transitions_displacy(transitions, tokenized_sentence)

buffer,stack,parse,action
ROOT Economic news had little effect on financial markets .,,,INIT
Economic news had little effect on financial markets .,ROOT,,shift
news had little effect on financial markets .,ROOT Economic,,shift
news had little effect on financial markets .,ROOT,,leftArc-amod
had little effect on financial markets .,ROOT news,,shift
had little effect on financial markets .,ROOT,,leftArc-nsubj
little effect on financial markets .,ROOT had,,rightArc-root
effect on financial markets .,ROOT had little,,shift
effect on financial markets .,ROOT had,,leftArc-amod
on financial markets .,ROOT had effect,,rightArc-dobj


Transition-based parsing converts graph prediction into a sequence of action classifications. A **complete sequence of legal actions** produces a dependency tree.

We can use a classifier such as logistic regression or a neural network to predict the next action from the current configuration. For example, with `saw` as the second stack item and `Bob` on top, a `RIGHT-ARC-dobj` action attaches `Bob` to `saw`. Word embeddings can reduce the sparsity of lexical features ([Chen and Manning, 2014](https://aclanthology.org/D14-1082/)).

The next question is where we get the training data to train the classifier, i.e. configurations labeled with the correct action. However the training data we are typically provided with consists of sentences labeled with the final dependency tree. Thus, we need to develop a function that given a sentence and its dependency tree can "reverse-engineer" the parsing process to recover the sequence of actions that was used construct it. This function is often referred to as the oracle (name inspired by [the ones in antiquity](https://en.wikipedia.org/wiki/Oracle)), and it is usually a set of heuristics that returns the correct sequence of actions by looking at the dependency tree. A different way to think about it is that of a human annotator demonstrating how to construct the parse tree using the transition system defined.

The implementation above is **arc-standard**. In the **arc-eager** system ([Nivre, 2003](https://aclanthology.org/W03-3017/)), arc actions connect the stack top and buffer front, and there is a separate REDUCE action. These systems use different action definitions and different oracles.

The basic arc-standard and arc-eager systems produce **projective** trees: when words, including `ROOT`, are laid out in sentence order, arcs do not cross.

Long-distance dependencies can require a non-projective analysis. In "What did economic news have little effect on?", `What` is the complement of the stranded preposition `on`. In the preposition-headed analysis below, `on` → `What` crosses `ROOT` → `have` (and `have` → `?`).

In [ ]:
tokens = ["ROOT", "What", "did", "economic", "news", "have", "little", "effect", "on", "?"]
arcs = {
    (0, 5, "root"),
    (5, 2, "aux"),
    (4, 3, "amod"),
    (5, 4, "nsubj"),
    (5, 7, "dobj"),
    (7, 6, "amod"),
    (7, 8, "prep"),
    (8, 1, "pobj"),
    (5, 9, "punct"),
}

transition.render_displacy(*transition.to_displacy_graph(arcs, tokens))

<Dependency diagram>

Non-projective trees require a transition system that permits crossing dependencies, for example one using multiple stacks ([Gómez-Rodríguez and Nivre, 2010](https://aclanthology.org/P10-1151/)).